# DATA ANALYTICS ML CHALLENGE · 2026-2
## Laboratorio 2 · Semana 14 · PCA, regularización y validación estadística

**Estudiante:** Sara Valentina Guauque Zarrate  
**ID oficial:** 17 · **Semilla:** 1017  
**Asignatura:** Data Analytics · Ingeniería de Sistemas y Computación  
**Universidad:** Universidad Central · **Profesor:** Elias Buitrago Bolivar

**Pregunta de negocio:** ¿Conviene un modelo que acierta más, identifica más retrasos o ocasiona menos costos? Para responder, comparo **cuatro modelos de regresión logística** sobre **la misma muestra real de Olist**. Conservo la partición de S12, sin usar el conjunto de prueba para escoger el modelo.

**Ruta de trabajo:** Carga → Preparación → PCA → Modelado A–D → Evaluación de métricas y costos → Kolmogorov-Smirnov → Conclusiones y evidencias.

**Referencia del trabajo:** Semana14_Olist.html y Lectura_Apoyo_S14.html, material de clase.


## 1. Carga de datos

Descargo los mismos siete CSV reales utilizados en S12, fijados al commit indicado por el profesor. No se suben archivos originales ni muestras al repositorio. Cada observación representa **un pedido**, aunque tenga varios ítems o pagos.


In [ ]:
import os
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 130)

from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from scipy.stats import ks_2samp
import sklearn
import sys

Path('reports/figuras').mkdir(parents=True,exist_ok=True)


In [ ]:
# Datos originales de la revision fija indicada por el profesor.
# El cambio opcional OLIST_DATA_DIR permite probar sin conexion si los CSV
# originales ya existen localmente. En Colab se usa el origen publico fijo.
ORIGEN_PUBLICO = ('https://raw.githubusercontent.com/spdrio/'
                  'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
                  'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')
ORIGEN = os.environ.get('OLIST_DATA_DIR', ORIGEN_PUBLICO).rstrip('/')

try:
    pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
    items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
    productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
    clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
    vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
    pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
    categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')
except Exception as exc:
    raise RuntimeError(
        'No se pudieron leer los siete CSV de Olist. Compruebe la conexion ' 
        'a internet en Colab o configure OLIST_DATA_DIR con los archivos originales.'
    ) from exc

for nombre, tabla in [('Pedidos', pedidos), ('Items', items),
                      ('Productos', productos), ('Clientes', clientes),
                      ('Vendedores', vendedores), ('Pagos', pagos),
                      ('Categorias', categorias)]:
    print(f'{nombre:12s}: {tabla.shape[0]:>7,} filas | {tabla.shape[1]} columnas')


## 2. Preparación y partición individual

Integro productos, vendedores, pagos y clientes sin duplicar pedidos. Las variables observadas después de la compra se usan solo para calcular el objetivo, no como predictores. Mantengo las **24 variables numéricas y 4 categóricas** de S12.


In [ ]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [ ]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

In [ ]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

In [ ]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

### Mismo ID, mismo conjunto, mismas particiones

La muestra se obtiene de forma estratificada con semilla 1000 + 17. La huella digital permite comprobar que la selección de pedidos es la misma que la de S12. La validación se utiliza para elegir entre A–D; los 5.250 pedidos de prueba se reservan para el cierre del desafío.


In [ ]:
# ID OFICIAL CONFIRMADO de Sara Valentina Guauque Zarrate.
# No modificar: todas las semanas deben conservar este identificador.
ID_ESTUDIANTE = 17
if not isinstance(ID_ESTUDIANTE, int) or isinstance(ID_ESTUDIANTE, bool) or not 1 <= ID_ESTUDIANTE <= 30:
    raise ValueError('Antes de ejecutar, escriba su ID OFICIAL (entero de 1 a 30) en ID_ESTUDIANTE.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

# Comprobaciones de individualización con la ejecución oficial de S12:
assert identificador == '17' and semilla == 1017
assert huella_muestra == '55ced66ce4f4', 'La muestra cambió respecto a S12: revisar origen, limpieza y particiones.'
assert (len(muestra),len(train),len(valid),len(test)) == (35000,24500,5250,5250)
assert len(columnas_numericas)==24 and len(columnas_categoricas)==4
assert not(set(train['order_id']) & set(valid['order_id']))
assert not(set(train['order_id']) & set(test['order_id']))
assert not(set(valid['order_id']) & set(test['order_id']))
print('Misma muestra S12 verificada. Huella:',huella_muestra)


## 3. PCA y SVD: varianza explicada acumulada

PCA transforma variables numéricas correlacionadas en componentes. Antes, la **imputación** y la **estandarización** se ajustan exclusivamente sobre los 24.500 pedidos de entrenamiento. Los datos de validación no intervienen para encontrar las componentes. Se retienen las necesarias para explicar **al menos el 90 % de la varianza numérica**; las cuatro variables categóricas siguen por su propia ruta de One-Hot Encoding.

El método se calcula mediante SVD. **El 90 % de varianza no equivale al 90 % de Accuracy**: solo indica cuánta variabilidad de las entradas se conserva.


In [ ]:
preparacion_numerica_exploratoria = make_pipeline(
    SimpleImputer(strategy='median'), StandardScaler()
)
numericas_train = preparacion_numerica_exploratoria.fit_transform(
    X_train[columnas_numericas]
)
pca_exploratorio = PCA(svd_solver='full')
pca_exploratorio.fit(numericas_train)

varianza_acumulada = np.cumsum(pca_exploratorio.explained_variance_ratio_)
componentes_90 = int(np.searchsorted(varianza_acumulada, 0.90) + 1)
varianza_retenida = float(varianza_acumulada[componentes_90 - 1])

assert len(varianza_acumulada) == len(columnas_numericas) == 24
assert varianza_retenida >= 0.90 - 1e-12
print(f'Variables numéricas originales: {len(columnas_numericas)}')
print(f'Componentes que explican al menos el 90 %: {componentes_90}')
print(f'Varianza numérica explicada: {varianza_retenida:.2%}')
print('Variables categóricas en ruta separada:', len(columnas_categoricas))

fig, ax = plt.subplots(figsize=(8.5, 4.7))
ax.plot(range(1, len(varianza_acumulada)+1),
        100 * varianza_acumulada, marker='o', markersize=3.5,
        linewidth=2, label='Varianza acumulada')
ax.axhline(90, color='#a65f45', linestyle='--', linewidth=1.5,
           label='Umbral 90 %')
ax.axvline(componentes_90, color='#4b5977', linestyle=':',
           label=f'{componentes_90} componentes')
ax.scatter([componentes_90], [varianza_retenida*100],
           color='#285f52', s=65, zorder=5)
ax.set(xlabel='Número de componentes', ylabel='Varianza explicada acumulada (%)',
       title='PCA · Entrenamiento ID 17', ylim=(0,103),
       xlim=(1,len(columnas_numericas)))
ax.set_xticks(range(1,len(columnas_numericas)+1,2))
ax.grid(alpha=0.18)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig('reports/figuras/S14_pca_varianza_ID_17.png',dpi=160,bbox_inches='tight')
plt.show()


## 4. Modelado: cuatro configuraciones controladas

El criterio del profesor exige utilizar los mismos datos, la misma ponderación de clase y el mismo umbral de 0,50; solamente cambiamos PCA y el parámetro de regularización C.

| Modelo | Numéricas | C | Regularización |
|---|---|---:|---|
| A | Sin PCA | 1,0 | Referencia |
| B | PCA 90 % | 1,0 | Referencia con PCA |
| C | PCA 90 % | 0,1 | Más fuerte |
| D | PCA 90 % | 10,0 | Más débil |

En las cuatro alternativas: class_weight='balanced'. Cuanto **menor C**, **mayor** penalización. La imputación, la escala, PCA y la regresión están en un Pipeline, de modo que solo se ajustan con entrenamiento.


In [ ]:
CONFIGURACIONES = (
    ('A_sin_PCA_C1', False, 1.0),
    ('B_PCA90_C1', True, 1.0),
    ('C_PCA90_C0_1', True, 0.1),
    ('D_PCA90_C10', True, 10.0),
)

def construir_modelo(utilizar_pca, c):
    pasos_numericos = [
        ('imputacion', SimpleImputer(strategy='median')),
        ('escalado', StandardScaler()),
    ]
    if utilizar_pca:
        pasos_numericos.append(
            ('pca', PCA(n_components=0.90, svd_solver='full'))
        )

    preparacion = ColumnTransformer([
        ('numericas', Pipeline(pasos_numericos), columnas_numericas),
        ('categoricas', Pipeline([
            ('imputacion', SimpleImputer(strategy='most_frequent')),
            ('codificacion', OneHotEncoder(handle_unknown='ignore')),
        ]), columnas_categoricas),
    ])
    return Pipeline([
        ('preparacion', preparacion),
        ('clasificador', LogisticRegression(
            C=c, class_weight='balanced', max_iter=1000,
            random_state=semilla, solver='lbfgs'
        )),
    ])

registros = []
modelos_entrenados = {}
for nombre, aplicar_pca, c in CONFIGURACIONES:
    modelo_actual = construir_modelo(aplicar_pca, c)
    modelo_actual.fit(X_train, y_train)
    pred_actual = modelo_actual.predict(X_valid)
    proba_actual = modelo_actual.predict_proba(X_valid)[:, 1]
    tn, fp, fn, tp = [int(x) for x in confusion_matrix(
        y_valid, pred_actual, labels=[0, 1]).ravel()]

    fila = {
        'modelo': nombre,
        'aplica_pca': bool(aplicar_pca),
        'C': float(c),
        'umbral': 0.50,
        'class_weight': 'balanced',
        'accuracy': float(accuracy_score(y_valid,pred_actual)),
        'precision': float(precision_score(y_valid,pred_actual,zero_division=0)),
        'recall': float(recall_score(y_valid,pred_actual,zero_division=0)),
        'f1': float(f1_score(y_valid,pred_actual,zero_division=0)),
        'ap': float(average_precision_score(y_valid,proba_actual)),
        'roc_auc': float(roc_auc_score(y_valid,proba_actual)),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'costo_fp_brl': 8 * fp, 'costo_fn_brl': 35 * fn,
        'costo_brl': 8 * fp + 35 * fn,
    }
    modelos_entrenados[nombre] = modelo_actual
    registros.append(fila)
    print(f"{nombre}: F1 {fila['f1']:.3f} | Recall {fila['recall']:.3f} "
          f"| AP {fila['ap']:.3f} | FP {fp} | FN {fn} | "
          f"Costo R$ {fila['costo_brl']:,}")

tabla_modelos = pd.DataFrame(registros).set_index('modelo')
display(tabla_modelos[['aplica_pca','C','accuracy','precision',
                      'recall','f1','ap','fp','fn','costo_brl']].round(4))
assert len(tabla_modelos) == 4
assert all(tabla_modelos['umbral'] == 0.5)
assert all(tabla_modelos['class_weight'] == 'balanced')
assert all(tabla_modelos['tn']+tabla_modelos['fp']+
           tabla_modelos['fn']+tabla_modelos['tp'] == 5250)
assert all(tabla_modelos['costo_brl'] == 8*tabla_modelos['fp']+
           35*tabla_modelos['fn'])

# El PCA dentro del modelo debe ser consistente con el análisis numérico previo.
for nombre in ('B_PCA90_C1', 'C_PCA90_C0_1', 'D_PCA90_C10'):
    pca_ajustado = modelos_entrenados[nombre].named_steps[
        'preparacion'].named_transformers_['numericas'].named_steps['pca']
    assert int(pca_ajustado.n_components_) == componentes_90
    assert float(np.sum(pca_ajustado.explained_variance_ratio_)) >= 0.90


## 5. Evaluación, costos y elección del modelo

**No selecciono por Accuracy solamente**, porque los retrasos representan una minoría. Comparo Precision, Recall, F1 y Average Precision. El costo ilustrativo de la actividad es:

**Costo = 8 × falsos positivos + 35 × falsos negativos** (reales brasileños).

Los costos son **supuestos académicos**, no facturación de Olist. Mantengo el umbral de 0,50 al comparar A–D para no confundir el efecto de PCA y regularización con el de cambiar la regla de decisión. La elección operativa prioriza el menor costo en validación, con F1 como desempate.


In [ ]:
# Comparación bajo una única regla empresarial.
orden_costo = tabla_modelos.sort_values(
    by=['costo_brl','f1','recall'], ascending=[True, False, False])
nombre_elegido = str(orden_costo.index[0])
resultado_elegido = orden_costo.iloc[0]
mejor_f1 = str(tabla_modelos['f1'].idxmax())

print('Mayor F1:', mejor_f1)
print('Menor costo ilustrativo:', nombre_elegido)
print('Costo del modelo elegido: R$ {:,}'.format(
    int(resultado_elegido['costo_brl'])))
print('Recordatorio: elegir un menor costo no implica superioridad estadística.')

fig, axes = plt.subplots(1, 2, figsize=(12.7, 4.6))
etiquetas = ['A\nSin PCA','B\nPCA C=1','C\nPCA C=0.1','D\nPCA C=10']
pos = np.arange(len(tabla_modelos))
ancho = 0.24
for j, metrica in enumerate(('f1','recall','ap')):
    axes[0].bar(pos + (j-1)*ancho, tabla_modelos[metrica].to_numpy(),
                width=ancho, label=metrica.upper())
axes[0].set_xticks(pos,etiquetas)
axes[0].set(title='Métricas de validación',ylabel='Proporción',ylim=(0,1))
axes[0].legend()
axes[0].grid(axis='y',alpha=0.18)

axes[1].bar(pos, tabla_modelos['costo_fp_brl'],label='Falsos positivos × R$ 8')
axes[1].bar(pos, tabla_modelos['costo_fn_brl'],
            bottom=tabla_modelos['costo_fp_brl'],label='Falsos negativos × R$ 35')
axes[1].set_xticks(pos,etiquetas)
axes[1].set(title='Costo didáctico por errores',ylabel='Reales (R$)')
axes[1].legend()
axes[1].grid(axis='y',alpha=0.18)
fig.suptitle('Olist ID 17 · Mismos 5.250 pedidos de validación')
fig.tight_layout()
fig.savefig('reports/figuras/S14_comparacion_modelos_ID_17.png',
            dpi=155,bbox_inches='tight')
plt.show()

tabla_modelos.reset_index().to_csv('reports/S14_experimentos_ID_17.csv',
                                   index=False,encoding='utf-8-sig')


## 6. Contraste estadístico KS sobre el valor del flete

Aplico la prueba **Kolmogorov–Smirnov de dos muestras** sobre valor_flete en entrenamiento y validación.

**H0:** el flete tiene la misma distribución poblacional en ambos grupos.  
**H1:** las distribuciones del flete son diferentes.  
**α = 0,05:** si p < 0,05, rechazo H0. Si no, no tengo evidencia suficiente para rechazarla.

El estadístico **D** es la mayor separación absoluta entre las distribuciones acumuladas. La prueba **no compara directamente los modelos**. Como el flete se registra con importes repetidos y redondeados, el valor p requiere cautela; tampoco prueba que los conjuntos sean equivalentes.


In [ ]:
flete_train = train['valor_flete'].dropna().astype(float).to_numpy()
flete_valid = valid['valor_flete'].dropna().astype(float).to_numpy()
ks = ks_2samp(flete_train, flete_valid, alternative='two-sided',
              method='asymp')
estadistico_d = float(ks.statistic)
valor_p = float(ks.pvalue)
alfa = 0.05
rechazar_h0 = bool(valor_p < alfa)
print(f'KS sobre valor_flete: D = {estadistico_d:.6f} | p = {valor_p:.6g}')
print('Tamaños:',len(flete_train), 'entrenamiento y',
      len(flete_valid),'validación')
print('Decisión al 5 %:', 'Rechazar H0' if rechazar_h0
      else 'No rechazar H0')
print('Nota: los empates en valores monetarios limitan la aproximación clásica.')

# Curvas de distribución acumulada para complementar el estadístico.
fig, ax = plt.subplots(figsize=(8.3, 4.7))
for datos,etiqueta in ((flete_train,'Entrenamiento'),
                       (flete_valid,'Validación')):
    ordenados = np.sort(datos)
    ecdf = np.arange(1,len(ordenados)+1)/len(ordenados)
    ax.step(ordenados,ecdf,where='post',label=f'{etiqueta} (n={len(datos):,})')
# Mostrar un rango útil de la distribución sin eliminar datos de la prueba.
limite = float(np.nanpercentile(np.r_[flete_train,flete_valid],99.5))
ax.set_xlim(0, max(1,limite))
ax.set(xlabel='Valor del flete (R$)',ylabel='Proporción acumulada',
       title=f'KS · D={estadistico_d:.4f} · p={valor_p:.4g}')
ax.legend()
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig('reports/figuras/S14_ks_flete_ID_17.png',
            dpi=155,bbox_inches='tight')
plt.show()


## 7. Autoverificación de reproducibilidad

Las siguientes aserciones comprueban las particiones, las cuatro configuraciones, la ausencia de datos de prueba en el ajuste, el criterio de PCA, el cálculo de costos y los resultados del contraste KS.


In [ ]:
assert len(muestra) == 35000 and (len(train),len(valid),len(test)) == (24500,5250,5250)
assert len(set(muestra['order_id']))==35000
assert huella_muestra == '55ced66ce4f4' and ID_ESTUDIANTE==17
assert all(col not in columnas for col in (
    'order_delivered_customer_date','dias_entrega','is_late'))
assert 1 <= componentes_90 <= 24 and varianza_retenida >= 0.90
assert set(tabla_modelos.index) == {v[0] for v in CONFIGURACIONES}
assert all(np.isfinite(tabla_modelos[['f1','ap','recall','costo_brl']].to_numpy()).ravel())
assert float(tabla_modelos['C'].loc['B_PCA90_C1']) == 1.0
assert float(tabla_modelos['C'].loc['C_PCA90_C0_1']) == 0.1
assert float(tabla_modelos['C'].loc['D_PCA90_C10']) == 10.0
assert 0 <= estadistico_d <= 1 and 0 <= valor_p <= 1
assert len(flete_train)>0 and len(flete_valid)>0
assert nombre_elegido in tabla_modelos.index
print('Autoverificación S14 superada: ID 17, 35.000 pedidos, 4 modelos, PCA, KS, costos.')


## 8. Conclusión de negocio

La interpretación de estos experimentos se construye a partir de los resultados de validación:

1. **PCA:** explicar la cantidad de variables iniciales, las componentes conservadas y si la reducción mejoró o empeoró Recall/F1 frente al modelo sin PCA.
2. **Comparación y costo:** justificar qué modelo se elegiría realmente considerando falsos positivos, falsos negativos y costo, no solo Accuracy.
3. **KS:** explicar D, p, la decisión estadística y por qué la prueba no certifica que las particiones sean idénticas.
4. **Límite de la conclusión:** el test reservado no se usó y los costos son supuestos del ejercicio.


### Resultados individuales

Los resultados cuantitativos y las conclusiones se registran en la sección final al ejecutar este notebook con los CSV originales y el ID 17.


## 9. Exportación de evidencias

Guardo un JSON con la huella de la muestra, los 4 modelos, métricas y costo; un CSV de experimentos; las gráficas PCA/modelos/KS y un informe ejecutivo en Markdown. Estos resultados se versionarán junto al notebook, **sin subir los CSV de Olist**.


In [ ]:
referencia_a = tabla_modelos.loc['A_sin_PCA_C1']
referencia_b = tabla_modelos.loc['B_PCA90_C1']
mejor_reg = tabla_modelos.loc[nombre_elegido]

texto_ks = (
    'Al 5 % se rechaza la hipótesis de una distribución común de fletes.'
    if rechazar_h0 else
    'Al 5 % no encontramos evidencia suficiente para rechazar la hipótesis '
    'de una distribución común de fletes. Esto no demuestra igualdad.'
)
cambio_f1 = 100*(referencia_b['f1']-referencia_a['f1'])
cambio_recall = 100*(referencia_b['recall']-referencia_a['recall'])

lineas = [
    '# Laboratorio 2 · Semana 14 · Olist · ID 17',
    '',
    '**Estudiante:** Sara Valentina Guauque Zarrate',
    '**Datos:** 35.000 pedidos reales; 24.500 train, 5.250 valid y 5.250 test reservado.',
    f'**Semilla:** {semilla} · **Huella:** {huella_muestra}',
    '',
    '## PCA',
    f'Partí de {len(columnas_numericas)} variables numéricas y PCA conservó '
    f'{componentes_90} componentes, con {varianza_retenida:.2%} de varianza explicada.',
    f'Al comparar B (PCA) con A (sin PCA), el cambio fue de '
    f'{cambio_f1:+.2f} puntos porcentuales en F1 y '
    f'{cambio_recall:+.2f} puntos en Recall. '
    'Conservar varianza no garantiza mejorar la detección de entregas tardías.',
    '',
    '## Cuatro modelos y decisión',
    'Todos los modelos usaron la misma validación, class_weight=balanced y umbral 0,50.',
    '',
    '| Modelo | Accuracy | Precision | Recall | F1 | AP | FP | FN | Costo R$ |',
    '|---|---:|---:|---:|---:|---:|---:|---:|---:|',
]
for nombre,fila in tabla_modelos.iterrows():
    lineas.append(
        f'| {nombre} | {fila["accuracy"]:.3f} | {fila["precision"]:.3f} '
        f'| {fila["recall"]:.3f} | {fila["f1"]:.3f} | {fila["ap"]:.3f} '
        f'| {int(fila["fp"])} | {int(fila["fn"])} | {int(fila["costo_brl"]):,} |'
    )
lineas.extend([
    '',
    f'Entre A–D, la alternativa con menor costo fue **{nombre_elegido}**, '
    f'con R$ {int(mejor_reg["costo_brl"]):,} calculados a partir de '
    f'{int(mejor_reg["fp"])} falsos positivos y {int(mejor_reg["fn"])} falsos negativos. '
    f'La de mayor F1 fue {mejor_f1}. La elección responde al criterio '
    'de costo de esta actividad, no a una superioridad estadística demostrada.',
    '',
    '## Prueba KS: flete entre entrenamiento y validación',
    f'**H0:** ambas distribuciones poblacionales son iguales. '
    f'**H1:** son diferentes. **α:** {alfa}.',
    f'El estadístico fue D={estadistico_d:.6f} y el valor p={valor_p:.6g}. {texto_ks}',
    'KS compara distribuciones de una variable, no desempeño de clasificadores. '
    'Los importes repetidos pueden afectar el valor p de la prueba clásica.',
    '',
    '## Alcance y limitaciones',
    'No se utilizó el conjunto de prueba para ajustar, escoger ni evaluar los modelos. '
    'El análisis incluye pedidos efectivamente entregados; los costos R$ 8 y R$ 35 '
    'son supuestos didácticos, no cifras contables de Olist. ',
])
informe = '\n'.join(lineas) + '\n'
Path('reports/S14_analisis_ID_17.md').write_text(informe,encoding='utf-8')

registro = {
    'laboratorio': 'S14_Lab2',
    'id_estudiante': '17',
    'semilla': int(semilla),
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'huella_muestra': huella_muestra,
    'n_muestra': int(len(muestra)),
    'n_train': int(len(train)),
    'n_valid': int(len(valid)),
    'n_test_reservado': int(len(test)),
    'n_variables_originales': int(len(columnas)),
    'n_numericas': int(len(columnas_numericas)),
    'n_categoricas': int(len(columnas_categoricas)),
    'n_componentes_pca_90': int(componentes_90),
    'varianza_retenida': float(varianza_retenida),
    'umbral': 0.50,
    'class_weight': 'balanced',
    'costo_fp_brl': 8,
    'costo_fn_brl': 35,
    'configuraciones': [dict(row) for row in registros],
    'modelo_menor_costo': nombre_elegido,
    'modelo_mayor_f1': mejor_f1,
    'ks': {
        'variable': 'valor_flete',
        'hipotesis_nula': 'mismas distribuciones',
        'estadistico_d': estadistico_d,
        'pvalue': valor_p,
        'alfa': alfa,
        'rechaza_h0': rechazar_h0,
        'n_train': int(len(flete_train)),
        'n_valid': int(len(flete_valid)),
        'limitacion': 'Importes redondeados y empates; no compara modelos.',
    },
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
}
archivo_json = Path('reports/S14_resultados_ID_17.json')
archivo_json.write_text(json.dumps(registro,indent=2,ensure_ascii=False),
                        encoding='utf-8')
print('Informes generados:', archivo_json, 'y reports/S14_analisis_ID_17.md')
print('Tablas y figuras conservadas en reports/')


---

### Reproducibilidad, interpretación y ética

- La generación de la muestra y el método son comparables con S12: ID 17, semilla 1017 y huella 55ced66ce4f4.
- PCA se aplica solamente a las 24 numéricas; los 4 campos categóricos usan One-Hot Encoding.
- Todos los transformadores se ajustan únicamente con entrenamiento; validación se utiliza para comparar.
- El conjunto de prueba se reserva para la evaluación final y no participa en estas decisiones.
- Los resultados de KS requieren cautela por los empates; no demuestran igualdad ni superioridad de un modelo.
- Los costos R$ 8 / R$ 35 son cifras didácticas del profesor, no costos reales medidos de Olist.
